# Week 12: Final demonstration and explanation

**Goal:** Demonstrate the policy assistant and explain its limits with evidence.

This notebook runs with the Python standard library. All records are fictional. Run cells top to bottom, then change one input and explain the observed result. Classroom stand-ins are labelled; they are not trained generative models.

## 1. Set up the case

A fresh student question and a previously unseen failure are presented.

In [ ]:
import re
policies = [{'id': 'P1', 'text': 'Student travel costs, including bus tickets, are not reimbursed.', 'role': 'student', 'version': '2027-01'}, {'id': 'P2', 'text': 'Library loans may be extended once for seven days.', 'role': 'student', 'version': '2027-01'}, {'id': 'P3', 'text': 'Students may request one deadline extension within seven days of the original due date.', 'role': 'student', 'version': '2027-01'}, {'id': 'P4', 'text': 'Academic appeals must be filed within ten days of the result.', 'role': 'student', 'version': '2027-01'}, {'id': 'P5', 'text': 'Staff payroll changes require manager approval before any record is changed.', 'role': 'staff', 'version': '2027-01'}]
def answer(question,role='student'):
    words=set(re.findall(r'[a-z]+',question.lower()))
    candidates=[p for p in policies if p['role']=='student' or role=='staff']
    scored=sorted(((len(words & set(re.findall(r'[a-z]+',p['text'].lower()))),p) for p in candidates),key=lambda x:x[0],reverse=True)
    if not scored or scored[0][0]<2: return {'status':'unsupported','text':None,'source':None}
    p=scored[0][1]
    return {'status':'extract','text':p['text'],'source':p['id']}
cases=[('academic appeals filed within ten days','student','P4'),
       ('library loans extended once','student','P2'),
       ('cafeteria lunch menu','student',None),
       ('staff payroll changes','student',None)]
for q,role,wanted in cases:
    result=answer(q,role)
    print(q,result,'expected source',wanted)
assert answer('staff payroll changes','student')['source'] is None


Present an answerable case, an unsupported case and an access case. This is an extractive demonstrator, not a trained language model. Diagnose one fresh near-match failure and propose a test.

## 3. Record your evidence

| Original setting | Your one change | Observed output | Explanation | Limitation |
|---|---|---|---|---|
| | | | | |

**Exit question:** A polished demonstration without tests establishes that the system is ready. Explain why this is wrong.

**Reading:** [Microsoft, Retrieval augmented generation](https://learn.microsoft.com/en-us/azure/foundry/concepts/retrieval-augmented-generation), [OpenAI, Evaluation best practices](https://developers.openai.com/api/docs/guides/evaluation-best-practices), [Anthropic, Building effective agents](https://www.anthropic.com/engineering/building-effective-agents)

## A fair week 1 comparison
Use the same questions and references. The old responses are supplied replay fixtures, not historical production measurements.

In [ ]:
import re
POLICIES = [{'id': 'P1', 'text': 'Student travel costs including bus tickets are not reimbursed.', 'role': 'student', 'version': '2027-01'}, {'id': 'P2', 'text': 'Library loans may be extended once for seven days.', 'role': 'student', 'version': '2027-01'}, {'id': 'P3', 'text': 'Students may request one deadline extension within seven days of the original due date.', 'role': 'student', 'version': '2027-01'}, {'id': 'P4', 'text': 'Academic appeals must be filed within ten days of the result.', 'role': 'student', 'version': '2027-01'}, {'id': 'P5', 'text': 'Staff payroll changes require manager approval before any record is changed.', 'role': 'staff', 'version': '2027-01'}]
STOP={'a','an','the','is','are','to','of','my','can','i','what','how','for','in','and'}
def tokens(text):return set(re.findall(r'[a-z]+',text.lower()))-STOP
def rank(question,role='student',k=1,corpus=None):
    if not isinstance(question,str) or not question.strip():raise ValueError('Question must be nonempty text')
    if role not in ('student','staff'):raise ValueError('Unknown trusted role')
    if k not in (1,2):raise ValueError('k must be 1 or 2')
    corpus=POLICIES if corpus is None else corpus
    allowed=[p for p in corpus if p['role']=='student' or role=='staff']
    scored=sorted([(len(tokens(question)&tokens(p['text'])),p) for p in allowed],key=lambda x:x[0],reverse=True)
    return [p for score,p in scored if score>=2][:k]
def answer(question,role='student',k=1,corpus=None):
    found=rank(question,role,k,corpus)
    return {'status':'extract' if found else 'unsupported','text':' '.join(p['text'] for p in found),
            'sources':[p['id'] for p in found],'version':'2027-01'}
def safe_trace(request_id,result):
    return {'request_id':request_id,'status':result['status'],'sources':result['sources'],'version':result['version']}


In [ ]:
common=[('student travel costs','P1'),('academic appeals filed','P4')]
baseline_sources=[None,'P4']
new_sources=[answer(q)['sources'] for q,_ in common]
old=sum(observed==expected for observed,(_,expected) in zip(baseline_sources,common))
new=sum(expected in observed for observed,(_,expected) in zip(new_sources,common))
print('Matched two-case baseline:',old,'/2; final:',new,'/2')
print('These tiny illustrative counts do not establish general deployment quality.')
for q,role in [('library loans extended','student'),('cafeteria menu','student'),('staff payroll changes','student')]:print(q,answer(q,role))
